In [1]:
# Install required package
!pip install nba_api

from nba_api.stats.endpoints import teamestimatedmetrics
import pandas as pd

# Fetch and display team names from TeamEstimatedMetrics
def test_team_names():
    try:
        # Fetch team estimated metrics data
        team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        )
        data = team_metrics.get_data_frames()[0]

        # Extract and display team names
        team_names = data['TEAM_NAME'].unique()
        print("Teams in TeamEstimatedMetrics:")
        print(team_names)

        # Test for specific team names
        team_to_test = "Atlanta Hawks"  # Change this to test other teams
        if team_to_test in team_names:
            print(f"'{team_to_test}' is found in the TeamEstimatedMetrics data.")
        else:
            print(f"'{team_to_test}' is NOT found in the TeamEstimatedMetrics data.")
    except Exception as e:
        print(f"Error fetching team metrics: {e}")

# Run the test
test_team_names()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 6.0 MB/s eta 0:00:00
Teams in TeamEstimatedMetrics:
['Memphis Grizzlies' 'Chicago Bulls' 'Atlanta Hawks' 'Washington Wizards'
 'Utah Jazz' 'Oklahoma City Thunder' 'Detroit Pistons'
 'Cleveland Cavaliers' 'Toronto Raptors' 'Denver Nuggets' 'Indiana Pacers'
 'San Antonio Spurs' 'New Orleans Pelicans' 'Dallas Mavericks'
 'Houston Rockets' 'Portland Trail Blazers' 'Milwaukee Bucks'
 'Golden State Warriors' 'Sacramento Kings' 'Charlotte Hornets'
 'Los Angeles Lakers' 'Philadelphia 76ers' 'Minnesota Timberwolves'
 'LA Clippers' 'Phoenix Suns' 'New York Knicks' 'Brooklyn Nets'
 'Orlando Magic' 'Miami Heat' 'Boston Celtics']
'Atlanta Hawks' is found in the TeamEstimatedMetrics data.


In [4]:
from nba_api.stats.static import teams

# Get all 30 teams as a list of dictionaries
nba_teams = teams.get_teams()

# Extract just the 'full_name' from each dictionary
team_names = [team['full_name'] for team in nba_teams]

print(team_names)


['Atlanta Hawks', 'Boston Celtics', 'Cleveland Cavaliers', 'New Orleans Pelicans', 'Chicago Bulls', 'Dallas Mavericks', 'Denver Nuggets', 'Golden State Warriors', 'Houston Rockets', 'Los Angeles Clippers', 'Los Angeles Lakers', 'Miami Heat', 'Milwaukee Bucks', 'Minnesota Timberwolves', 'Brooklyn Nets', 'New York Knicks', 'Orlando Magic', 'Indiana Pacers', 'Philadelphia 76ers', 'Phoenix Suns', 'Portland Trail Blazers', 'Sacramento Kings', 'San Antonio Spurs', 'Oklahoma City Thunder', 'Toronto Raptors', 'Utah Jazz', 'Memphis Grizzlies', 'Washington Wizards', 'Detroit Pistons', 'Charlotte Hornets']


In [5]:
import requests

# 1. Configuration
API_KEY = 'f098da95884f42852df5ff12f5704741'  # Replace with your actual Odds API key
SPORT = 'basketball_nba'  # Sport key for NBA
REGIONS = 'us'            # Optional: regions to include (us, uk, au, eu)
MARKETS = 'h2h'           # Moneyline market typically contains team names

# 2. API Request
url = f'https://api.the-odds-api.com/v4/sports/{SPORT}/odds'
params = {
    'api_key': API_KEY,
    'regions': REGIONS,
    'markets': MARKETS,
}

response = requests.get(url, params=params)

# 3. Processing the response
if response.status_code == 200:
    games = response.json()

    # Use a set to store unique team names automatically
    nba_teams = set()

    for game in games:
        nba_teams.add(game['home_team'])
        nba_teams.add(game['away_team'])

    # Print the sorted list of names
    print(f"Found {len(nba_teams)} unique teams in upcoming games:")
    for team in sorted(nba_teams):
        print(f"- {team}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)


Found 12 unique teams in upcoming games:
- Boston Celtics
- Denver Nuggets
- Golden State Warriors
- Houston Rockets
- Los Angeles Clippers
- Milwaukee Bucks
- New York Knicks
- Philadelphia 76ers
- San Antonio Spurs
- Toronto Raptors
- Utah Jazz
- Washington Wizards


In [16]:
import requests

# 1. Configuration
API_KEY = 'f098da95884f42852df5ff12f5704741'
SPORT = 'basketball_nba'

# 2. Endpoint: Use the 'odds' endpoint to find active teams and their event IDs
# The Odds API returns 'id' at the game (event) level, not a static team directory
url = f'https://api.the-odds-api.com/v4/sports/{SPORT}/odds'
params = {
    'apiKey': API_KEY,
    'regions': 'us',
    'markets': 'h2h'
}

print(f"Fetching active NBA events for {SPORT}...")
response = requests.get(url, params=params)

if response.status_code == 200:
    events = response.json()

    # Store results in a list of unique team entries
    # Note: Event IDs are unique to the game, not the team itself.
    team_data = []
    seen_teams = set()

    for event in events:
        event_id = event.get('id')
        home_team = event.get('home_team')
        away_team = event.get('away_team')

        if home_team not in seen_teams:
            team_data.append({'team': home_team, 'recent_event_id': event_id})
            seen_teams.add(home_team)

        if away_team not in seen_teams:
            team_data.append({'team': away_team, 'recent_event_id': event_id})
            seen_teams.add(away_team)

    # 3. Output
    print(f"\nTeams found in current window: {len(team_data)}")
    for item in sorted(team_data, key=lambda x: x['team']):
        print(f"Team: {item['team']:<25} | Event ID: {item['recent_event_id']}")

else:
    print(f"Error {response.status_code}: {response.text}")


Fetching active NBA events for basketball_nba...

Teams found in current window: 12
Team: Boston Celtics            | Event ID: 3a1ef5b1bd5afc1ee417ba3992167357
Team: Denver Nuggets            | Event ID: 1804ec1ea561b137f3b6a151af196bef
Team: Golden State Warriors     | Event ID: 48deab86651d256447d6bb35a3d9939c
Team: Houston Rockets           | Event ID: 23f7e8ebef76ec1b639f412eccfdfda4
Team: Los Angeles Clippers      | Event ID: 48deab86651d256447d6bb35a3d9939c
Team: Milwaukee Bucks           | Event ID: 3a1ef5b1bd5afc1ee417ba3992167357
Team: New York Knicks           | Event ID: 6e039621e88bb0b022d5b0d3eafd9596
Team: Philadelphia 76ers        | Event ID: fe9fbae6e69b1684995b2fa8f45fb691
Team: San Antonio Spurs         | Event ID: fe9fbae6e69b1684995b2fa8f45fb691
Team: Toronto Raptors           | Event ID: 6e039621e88bb0b022d5b0d3eafd9596
Team: Utah Jazz                 | Event ID: 1804ec1ea561b137f3b6a151af196bef
Team: Washington Wizards        | Event ID: 23f7e8ebef76ec1b639f412ec